# Hermitian Sparse Matrix Block Encoding

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389.
Copyright Google LLC. SPDX-License-Identifier: Apache-2.0.

In [ ]:
from collections import Counter

import torch
from IPython.display import Markdown, SVG, display
from sympy import Symbol

from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum import Circuit
from flagquantum.compiler._component_trace.resource_model import estimate_component_resources
from flagquantum.core.ir.program.register_types import opaque_register_type
from flagquantum.core.ir.program.registers import Register
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import (
    export_call_graph_dot, format_call_graph_markdown, render_call_graph_svg,
)
from flagquantum.simulation.unitary import get_unitary

def signature(width):
    return Signature((
        Register("system", opaque_register_type(width)),
        Register("ancilla", opaque_register_type(width + 2)),
    ))

## Hermitian sparse-access matrix

With equal row and column sparsity $s$, the block encodes $A/s$ for a
Hermitian sparse-access matrix $A$. The ancilla contains a flag, an
$n$-qubit ordinal and a second flag. The entry oracle rotates the second
flag with zero amplitude $\sqrt{A_{ij}}$.

Prepare the ordinal, apply the column and square-root entry oracles, swap
the index words and swap the flags, then apply the inverse preparation
chain. Only the two swaps acquire an outer positive control. The
unconditional forward and inverse stages cancel in the inactive sector.
The composed block is self-adjoint.

Pad irregular dimensions to powers of two and pad oracle lists with zero
entries to make their lengths equal. Symbolic analysis preserves the
declared shape and resource model. Executable binding uses concrete widths.

[Lecture Notes on Quantum Algorithms for Scientific Computation](https://arxiv.org/abs/2201.08309)
gives the original conjugated-swap construction.

In [ ]:
from flagquantum.algorithms.primitives.hermitian_sparse_matrix_block_encoding import (
    HermitianSparseMatrixBlockEncoding,
    UniformSqrtEntryOracle,
    source_uniform_hermitian_sparse_resource_formula,
)
from flagquantum.algorithms.primitives.sparse_matrix_oracles import TopLeftRowColumnOracle

### Example Instances

In [ ]:
n = Symbol("n", positive=True, integer=True)
symbolic_signature = signature(n)
symbolic_cost = source_uniform_hermitian_sparse_resource_formula(n)
print("Symbolic normalization", 2**n)

In [ ]:
column = TopLeftRowColumnOracle(2)
entry = UniformSqrtEntryOracle(2, 0.3)
encoding = HermitianSparseMatrixBlockEncoding(column, entry, eps=0)
assert (encoding.alpha, encoding.ancilla_bitsize) == (4, 4)

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([
    ("Symbolic Hermitian sparse block", symbolic_signature, {}),
    ("Concrete Hermitian sparse block", signature(encoding.system_bitsize), {}),
])))

### Call Graph

In [ ]:
root = "Hermitian sparse block"
leaves = dict.fromkeys((
    "Always prepare uniform", "Always column oracle", "Always square-root entry",
    "Swap n index bits", "Swap flags", "Always square-root entry adjoint",
    "Always column adjoint", "Always unprepare uniform",
), 1)
edges = {(root, child): count for child, count in leaves.items()}
display(SVG(render_call_graph_svg(export_call_graph_dot(root, edges))))
display(Markdown(format_call_graph_markdown(root, edges, leaves)))
print("Symbolic source resource formula", symbolic_cost)

### Append and verify the original uniform matrix

Project onto both zero flags and the zero ordinal to recover the constant
four-by-four matrix. The full circuit equals its adjoint, so appending it
again restores the entire six-qubit state space.

In [ ]:
circuit = Circuit(6, dtype=torch.complex128)
operation = encoding.make_on(system=(0, 1), ancilla=(2, 3, 4, 5))
circuit.append(operation)
matrix = get_unitary(circuit)
selected = torch.arange(4) * 16
torch.testing.assert_close(encoding.alpha * matrix[selected][:, selected],
                          torch.full((4, 4), 0.3, dtype=torch.complex128),
                          atol=2e-12, rtol=0)
print("Emitted primitives", dict(Counter(gate.name for gate in circuit.to_ir().instructions)))
print("Source resource model", estimate_component_resources(encoding.resource_model()))
circuit.append(operation)
torch.testing.assert_close(get_unitary(circuit), torch.eye(64, dtype=torch.complex128),
                           atol=2e-12, rtol=0)